# End-to-end Data Engineering + Analytics project


In [ ]:
# Stack: Python, Pandas, NumPy, SQLite, SQL, Matplotlib

!pip install ucimlrepo --quiet

import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo


# 1. SETUP & INGESTION


In [ ]:
# ============================================================
DATA_DIR = "/content/data"
RAW_CSV = f"{DATA_DIR}/online_retail_raw.csv"
DB_PATH = "/content/retail_sales.db"
os.makedirs(DATA_DIR, exist_ok=True)

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.original.copy()

df.to_csv(RAW_CSV, index=False)
print("Raw dataset:", df.shape)
print("Columns:", list(df.columns))


# 2. INSPECTION & QUALITY CHECKS


In [ ]:
# ============================================================
print("\n===== INSPECTION =====")
print(df.head())
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

print("\n===== QUALITY COUNTS =====")
print("Cancellation rows:", df["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Negative quantity rows:", (df["Quantity"] < 0).sum())
print("Invalid price rows:", (df["UnitPrice"] <= 0).sum())


# 3. DATABASE INITIALIZATION


In [ ]:
# ============================================================
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
cursor = conn.cursor()

# Recreate project tables so the pipeline is repeatable and
# cannot accidentally duplicate the Sales fact table.
cursor.executescript("""
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS country;
DROP TABLE IF EXISTS cleaned_retail;
DROP TABLE IF EXISTS raw_retail;

CREATE TABLE products (
    ProductKey INTEGER PRIMARY KEY AUTOINCREMENT,
    StockCode TEXT NOT NULL UNIQUE,
    Description TEXT
);

CREATE TABLE customers (
    CustomerKey INTEGER PRIMARY KEY AUTOINCREMENT,
    CustomerID INTEGER UNIQUE
);

CREATE TABLE country (
    CountryKey INTEGER PRIMARY KEY AUTOINCREMENT,
    Country TEXT NOT NULL UNIQUE
);

CREATE TABLE sales (
    SalesKey INTEGER PRIMARY KEY AUTOINCREMENT,
    InvoiceNo TEXT,
    ProductKey INTEGER,
    CustomerKey INTEGER,
    CountryKey INTEGER,
    InvoiceDate TEXT,
    Quantity INTEGER,
    UnitPrice REAL,
    FOREIGN KEY (ProductKey) REFERENCES products(ProductKey),
    FOREIGN KEY (CustomerKey) REFERENCES customers(CustomerKey),
    FOREIGN KEY (CountryKey) REFERENCES country(CountryKey)
);
""")
conn.commit()


# 4. RAW DATA -> SQLITE


In [ ]:
# ============================================================
df.to_sql("raw_retail", conn, if_exists="replace", index=False)

quality_sql = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN InvoiceNo LIKE 'C%' THEN 1 ELSE 0 END) AS cancellation_rows,
    SUM(CASE WHEN Quantity < 0 THEN 1 ELSE 0 END) AS negative_quantity_rows,
    SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END) AS invalid_price_rows,
    SUM(CASE WHEN InvoiceNo IS NULL THEN 1 ELSE 0 END) AS missing_invoice,
    SUM(CASE WHEN StockCode IS NULL THEN 1 ELSE 0 END) AS missing_stockcode,
    SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS missing_description,
    SUM(CASE WHEN Quantity IS NULL THEN 1 ELSE 0 END) AS missing_quantity,
    SUM(CASE WHEN InvoiceDate IS NULL THEN 1 ELSE 0 END) AS missing_invoice_date,
    SUM(CASE WHEN UnitPrice IS NULL THEN 1 ELSE 0 END) AS missing_unit_price,
    SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS missing_customer,
    SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS missing_country
FROM raw_retail;
"""
print("\n===== SQL QUALITY REPORT =====")
print(pd.read_sql_query(quality_sql, conn).T)


# 5. CLEANING


In [ ]:
# ============================================================
cursor.execute("""
CREATE TABLE cleaned_retail AS
SELECT *
FROM raw_retail
WHERE InvoiceNo NOT LIKE 'C%'
  AND Quantity > 0
  AND UnitPrice > 0;
""")
conn.commit()

counts = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM raw_retail) AS raw_rows,
    (SELECT COUNT(*) FROM cleaned_retail) AS cleaned_rows;
""", conn)
counts["rows_removed"] = counts["raw_rows"] - counts["cleaned_rows"]
print("\n===== CLEANING RESULT =====")
print(counts)


# 6. CURATED STAR SCHEMA


In [ ]:
# ============================================================
cursor.execute("""
INSERT INTO products (StockCode, Description)
SELECT StockCode, MAX(Description)
FROM cleaned_retail
WHERE StockCode IS NOT NULL
GROUP BY StockCode;
""")

cursor.execute("""
INSERT INTO customers (CustomerID)
SELECT DISTINCT CAST(CustomerID AS INTEGER)
FROM cleaned_retail
WHERE CustomerID IS NOT NULL;
""")

cursor.execute("""
INSERT INTO country (Country)
SELECT DISTINCT Country
FROM cleaned_retail
WHERE Country IS NOT NULL;
""")

cursor.execute("""
INSERT INTO sales (
    InvoiceNo, ProductKey, CustomerKey, CountryKey,
    InvoiceDate, Quantity, UnitPrice
)
SELECT
    r.InvoiceNo,
    p.ProductKey,
    c.CustomerKey,
    co.CountryKey,
    r.InvoiceDate,
    r.Quantity,
    r.UnitPrice
FROM cleaned_retail r
LEFT JOIN products p ON r.StockCode = p.StockCode
LEFT JOIN customers c ON CAST(r.CustomerID AS INTEGER) = c.CustomerID
LEFT JOIN country co ON r.Country = co.Country;
""")
conn.commit()

print("\n===== STAR SCHEMA VALIDATION =====")
print(pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM cleaned_retail) AS cleaned_rows,
    (SELECT COUNT(*) FROM sales) AS sales_rows,
    (SELECT COUNT(*) FROM products) AS product_rows,
    (SELECT COUNT(*) FROM customers) AS customer_rows,
    (SELECT COUNT(*) FROM country) AS country_rows;
""", conn))

print("\n===== FOREIGN KEY CHECK =====")
print(pd.read_sql_query("""
SELECT
    SUM(CASE WHEN ProductKey IS NULL THEN 1 ELSE 0 END) AS missing_product_keys,
    SUM(CASE WHEN CustomerKey IS NULL THEN 1 ELSE 0 END) AS missing_customer_keys,
    SUM(CASE WHEN CountryKey IS NULL THEN 1 ELSE 0 END) AS missing_country_keys
FROM sales;
""", conn))


# 7. SQL ANALYTICS


In [ ]:
# ============================================================
overall_kpis = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_orders,
    SUM(order_quantity) AS total_quantity_sold,
    SUM(order_revenue) AS total_revenue,
    AVG(order_revenue) AS average_order_value
FROM (
    SELECT
        InvoiceNo,
        SUM(Quantity) AS order_quantity,
        SUM(Quantity * UnitPrice) AS order_revenue
    FROM sales
    GROUP BY InvoiceNo
);
""", conn)
print("\n===== OVERALL KPIs =====")
print(overall_kpis)

product_analysis = pd.read_sql_query("""
SELECT
    p.StockCode,
    p.Description,
    SUM(s.Quantity) AS quantity_sold,
    SUM(s.Quantity * s.UnitPrice) AS revenue
FROM sales s
JOIN products p ON s.ProductKey = p.ProductKey
GROUP BY p.StockCode, p.Description
ORDER BY revenue DESC;
""", conn)

country_revenue = pd.read_sql_query("""
SELECT
    co.Country,
    SUM(s.Quantity * s.UnitPrice) AS revenue
FROM sales s
JOIN country co ON s.CountryKey = co.CountryKey
GROUP BY co.Country
ORDER BY revenue DESC;
""", conn)

# Window function example
ranked_products = pd.read_sql_query("""
WITH product_revenue AS (
    SELECT
        p.StockCode,
        p.Description,
        SUM(s.Quantity * s.UnitPrice) AS revenue
    FROM sales s
    JOIN products p ON s.ProductKey = p.ProductKey
    GROUP BY p.StockCode, p.Description
)
SELECT
    StockCode,
    Description,
    revenue,
    RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
FROM product_revenue
ORDER BY revenue_rank
LIMIT 10;
""", conn)
print("\n===== TOP REVENUE ENTRIES =====")
print(ranked_products)


# 8. MONTHLY ANALYSIS


In [ ]:
# ============================================================
sales_dates = pd.read_sql_query(
    "SELECT InvoiceDate, Quantity, UnitPrice FROM sales;", conn
)
sales_dates["InvoiceDate"] = pd.to_datetime(
    sales_dates["InvoiceDate"], format="%m/%d/%Y %H:%M"
)
sales_dates["revenue"] = sales_dates["Quantity"] * sales_dates["UnitPrice"]
sales_dates["sales_month"] = sales_dates["InvoiceDate"].dt.to_period("M").astype(str)

monthly_revenue = (
    sales_dates.groupby("sales_month", as_index=False)["revenue"].sum()
)
highest_month = monthly_revenue.loc[monthly_revenue["revenue"].idxmax()]
lowest_month = monthly_revenue.loc[monthly_revenue["revenue"].idxmin()]


# 9. PYTHON + NUMPY ANALYSIS


In [ ]:
# ============================================================
mean_product_revenue = product_analysis["revenue"].mean()
median_product_revenue = product_analysis["revenue"].median()
revenue_percentiles = np.percentile(
    product_analysis["revenue"].to_numpy(), [25, 50, 75, 90, 95]
)

total_product_revenue = product_analysis["revenue"].sum()
top_10_share = product_analysis.head(10)["revenue"].sum() / total_product_revenue * 100

highest_product = product_analysis.loc[product_analysis["revenue"].idxmax()]
highest_quantity_entry = product_analysis.loc[product_analysis["quantity_sold"].idxmax()]
highest_country = country_revenue.loc[country_revenue["revenue"].idxmax()]
highest_country_share = highest_country["revenue"] / country_revenue["revenue"].sum() * 100


# 10. VISUALIZATION


In [ ]:
# ============================================================
plt.figure(figsize=(11, 5))
plt.plot(monthly_revenue["sales_month"], monthly_revenue["revenue"], marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for title, data, column, label in [
    ("Top 10 Revenue Entries", product_analysis.head(10).sort_values("revenue"), "revenue", "Revenue"),
    ("Top 10 Countries by Revenue", country_revenue.head(10).sort_values("revenue"), "revenue", "Revenue"),
    ("Top 10 Entries by Quantity Sold", product_analysis.sort_values("quantity_sold").tail(10), "quantity_sold", "Quantity Sold")
]:
    plt.figure(figsize=(10, 6))
    name_col = "Description" if "Description" in data.columns else "Country"
    plt.barh(data[name_col], data[column])
    plt.title(title)
    plt.xlabel(label)
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(10, 5))
plt.hist(product_analysis["revenue"], bins=40)
plt.title("Product Revenue Distribution")
plt.xlabel("Revenue")
plt.ylabel("Number of Entries")
plt.tight_layout()
plt.show()


# 11. BUSINESS INSIGHTS


In [ ]:
# ============================================================
print("\n" + "=" * 60)
print("STAGE 11 — BUSINESS INSIGHTS")
print("=" * 60)
print(f"Total orders: {int(overall_kpis.loc[0, 'total_orders']):,}")
print(f"Total quantity sold: {int(overall_kpis.loc[0, 'total_quantity_sold']):,}")
print(f"Total revenue: {overall_kpis.loc[0, 'total_revenue']:,.2f}")
print(f"Average order value: {overall_kpis.loc[0, 'average_order_value']:,.2f}")
print(f"Highest revenue month: {highest_month['sales_month']}")
print(f"Lowest revenue month: {lowest_month['sales_month']}")
print(f"Highest revenue entry: {highest_product['Description']}")
print(f"Highest revenue country: {highest_country['Country']}")
print(f"Highest country revenue share: {highest_country_share:.2f}%")
print(f"Mean product revenue: {mean_product_revenue:,.2f}")
print(f"Median product revenue: {median_product_revenue:,.2f}")
print(f"Top 10 revenue-entry share: {top_10_share:.2f}%")


# 12. FINAL VALIDATION


In [ ]:
# ============================================================
final_validation = pd.read_sql_query("""
SELECT
    COUNT(*) AS sales_rows,
    COUNT(DISTINCT InvoiceNo) AS distinct_orders,
    SUM(Quantity) AS total_quantity,
    SUM(Quantity * UnitPrice) AS total_revenue
FROM sales;
""", conn)

print("\n===== FINAL VALIDATION =====")
print(final_validation)

conn.close()
print("\nPipeline completed successfully.")
